# 003 · Environments and Dependencies with uv

> **Syntax › Modules & Packages** · Python 3.14

In .NET, `dotnet restore` pulls NuGet packages into a shared cache and each project just *references* the versions it needs. Python installs packages **into an environment**, so every project needs its own isolated **virtual environment**. `uv` is a fast, all-in-one tool that manages Python versions, virtual environments, dependencies and lock files — the closest thing Python has to the `dotnet` CLI.

**In this notebook you will learn:**

- what a virtual environment is, and why Python needs one when .NET doesn't
- the `uv` workflow and how each command maps to the `dotnet` CLI
- how `pyproject.toml`, `uv.lock` and `.python-version` fit together
- dependency groups (dev-only packages)
- how to inspect the current environment from Python code

## C# mental model

| .NET | Python + uv |
|---|---|
| .NET SDK / runtime version | Python interpreter version (`uv python install`) |
| `global.json` (pin SDK version) | `.python-version` |
| `.csproj` | `pyproject.toml` |
| `packages.lock.json` | `uv.lock` |
| nuget.org | PyPI (pypi.org) |
| NuGet global packages folder (`~/.nuget/packages`) | uv's global cache — linked into each environment |
| `bin/` + `obj/` per project | `.venv/` per project (the virtual environment) |
| `dotnet` CLI | `uv` CLI |

The key mental shift: a .NET project's dependencies are resolved at build time and copied next to your assembly. In Python, `import` looks for packages in the **active environment's** `site-packages` folder at runtime *(see Modules & Packages › 001 · Importing Modules)* — so the environment *is* the dependency set.

## Why virtual environments?

Without isolation, every project on your machine would share one global `site-packages`, and project A needing `somelib 1.x` would clash with project B needing `somelib 2.x`.

A **virtual environment** is a folder (by convention `.venv/` in the project root) that contains:

- a link to a Python interpreter
- its own `site-packages` with exactly this project's dependencies
- a `pyvenv.cfg` file pointing back to the base Python installation

```text
my-shop/
├── .venv/                      # never commit this (like bin/ and obj/)
│   ├── bin/python              # Scripts\python.exe on Windows
│   ├── lib/python3.14/site-packages/
│   └── pyvenv.cfg
├── .python-version
├── pyproject.toml
└── uv.lock
```

With uv you rarely touch `.venv` directly: `uv run` and `uv sync` create and update it automatically. (Classic tools you'll see in older docs: `python -m venv .venv` and `pip install ...`.)

## Inspecting the current environment

This notebook's kernel runs inside the project's `.venv`. `sys.prefix` points at the active environment and `sys.base_prefix` at the underlying Python installation — if they differ, you are in a virtual environment.

In [ ]:
import sys
from pathlib import Path


def short(path: str) -> str:
    """Show paths relative to the home folder (keeps output machine-neutral)."""
    return path.replace(str(Path.home()), "~")


print("python version:", sys.version_info[:3])
print("executable:    ", short(sys.executable))
print("prefix:        ", short(sys.prefix))
print("base prefix:   ", short(sys.base_prefix))
print("in a venv?     ", sys.prefix != sys.base_prefix)

`sys.version_info` is a named tuple, so version checks are easy comparisons (≈ checking `Environment.Version` or using `#if NET8_0_OR_GREATER`):

In [ ]:
print(sys.version_info >= (3, 14))
print(sys.version_info.major, sys.version_info.minor)
print(sys.implementation.name)        # 'cpython' — the reference implementation

## Which packages are installed?

`importlib.metadata` reads the metadata of installed distributions — like inspecting the `project.assets.json` that `dotnet restore` produces.

In [ ]:
from importlib import metadata

for name in ("ipykernel", "jupyterlab"):
    print(f"{name:<12} {metadata.version(name)}")

try:
    metadata.version("surely-not-installed-package")
except metadata.PackageNotFoundError as e:
    print(f"{type(e).__name__}: {e}")

In [ ]:
dists = sorted(metadata.distributions(), key=lambda d: d.metadata["Name"].lower())
print(f"{len(dists)} distributions installed in this environment, e.g.:")
for d in dists[:5]:
    print(f"  {d.metadata['Name']} {d.version}")

ipykernel = metadata.distribution("ipykernel")
print(ipykernel.metadata["Summary"])
print("first requirements:", ipykernel.requires[:3])   # its own dependencies

💡 Most of those distributions are **transitive** dependencies of `jupyterlab` and `ipykernel` — the project only declares two direct dependencies (see the `pyproject.toml` below). `uv tree` shows the full graph, like `dotnet list package --include-transitive`.

## `pyproject.toml` ≈ `.csproj`

`pyproject.toml` is the standard project file: metadata, the supported Python version and dependencies. Here is the actual file for **this** course project:

```toml
[project]
name = "python-fundamentals"
version = "0.1.0"
requires-python = ">=3.14"
dependencies = [
    "ipykernel>=7.4.0",
    "jupyterlab>=4.6.4",
]

[dependency-groups]
dev = [
    "nbconvert>=7.17.1",
    "nbformat>=5.11.1",
]
```

The rough .NET equivalent:

```xml
<Project Sdk="Microsoft.NET.Sdk">
  <PropertyGroup>
    <TargetFramework>net9.0</TargetFramework>
    <Version>0.1.0</Version>
  </PropertyGroup>
  <ItemGroup>
    <PackageReference Include="SomeRuntimeLib" Version="7.4.0" />
    <PackageReference Include="SomeBuildTool" Version="7.17.1" PrivateAssets="all" />
  </ItemGroup>
</Project>
```

| `pyproject.toml` | `.csproj` |
|---|---|
| `name`, `version` | `<PackageId>`, `<Version>` |
| `requires-python = ">=3.14"` | `<TargetFramework>` |
| `dependencies = [...]` | `<PackageReference>` items |
| `"ipykernel>=7.4.0"` (a version **range**) | `Version="7.4.0"` (NuGet also treats this as "7.4.0 or higher") |
| `[dependency-groups] dev = [...]` | dev-only references (`PrivateAssets="all"`, test projects) |

The file is TOML, and Python can read TOML with the standard-library `tomllib` module (read-only):

In [ ]:
import tomllib

sample = """
[project]
name = "python-fundamentals"
requires-python = ">=3.14"
dependencies = ["ipykernel>=7.4.0", "jupyterlab>=4.6.4"]

[dependency-groups]
dev = ["nbconvert>=7.17.1", "nbformat>=5.11.1"]
"""

config = tomllib.loads(sample)
print(config["project"]["dependencies"])
print(config["dependency-groups"]["dev"])

## `uv.lock` ≈ `packages.lock.json`

`pyproject.toml` states **ranges** (`>=7.4.0`); `uv.lock` records the **exact** resolved version (plus hashes) of every direct and transitive package, for every platform. Commit it to source control so every developer and CI run gets an identical environment — the same role as `packages.lock.json` with `RestoreLockedMode`.

```toml
# an excerpt of uv.lock — generated, never edited by hand
[[package]]
name = "anyio"
version = "4.15.1"
source = { registry = "https://pypi.org/simple" }
dependencies = [
    { name = "idna" },
    ...
]
```

## `.python-version` ≈ `global.json`

A one-line file that pins the interpreter version for the project. This project's file contains:

```text
3.14
```

uv reads it to decide which Python to use — and downloads that version automatically if it isn't installed yet. `uv python pin 3.14` writes the file for you.

## The uv ↔ dotnet command map

| Task | .NET | uv |
|---|---|---|
| Create a new app project | `dotnet new console` | `uv init` (app) / `uv init --lib` (library, `src` layout) |
| Add a package | `dotnet add package X` | `uv add X` |
| Add a specific version | `dotnet add package X --version 2.1.0` | `uv add "X==2.1.0"` (or a range: `"X>=2,<3"`) |
| Add a dev-only package | `PrivateAssets="all"` / test project | `uv add --dev X` (→ `dev` group) / `uv add --group test X` |
| Remove a package | `dotnet remove package X` | `uv remove X` |
| Restore dependencies | `dotnet restore` | `uv sync` |
| Restore exactly from the lock file | `dotnet restore --locked-mode` | `uv sync --locked` |
| Update the lock file | (re-restore) | `uv lock` |
| Upgrade packages | `dotnet list package --outdated`, then edit versions | `uv lock --upgrade` / `uv lock --upgrade-package X` |
| Run the project / a script | `dotnet run` | `uv run python main.py` / `uv run main.py` |
| Show dependency graph | `dotnet list package --include-transitive` | `uv tree` |
| Install a global CLI tool | `dotnet tool install -g X` | `uv tool install X` |
| Run a tool once without installing | `dotnet tool exec X` / `dnx X` (.NET 10) | `uvx X` (= `uv tool run X`) |
| Install a runtime/SDK | download the .NET SDK | `uv python install 3.14` |
| Pin the runtime version | `global.json` | `uv python pin 3.14` → `.python-version` |
| Format code | `dotnet format` | `uv format` |
| Build a package | `dotnet pack` | `uv build` → `.whl` + `.tar.gz` |
| Publish a package | `dotnet nuget push` | `uv publish` |

💡 `uv run <command>` makes sure the environment is in sync with `uv.lock` (creating `.venv` if needed) and then runs the command inside it — no manual "activate" step. For example, `uv run jupyter lab` starts JupyterLab from this project's environment.

⚠️ **Gotcha:** don't run `pip install` into a uv-managed project. It changes `.venv` without updating `pyproject.toml` or `uv.lock`, so the next `uv sync` removes it again. Use `uv add` instead. (`uv pip install` exists for pip-style workflows outside of projects.)

## A typical workflow

These are **shell** commands — run them in a terminal, not in this notebook (running them here would modify the course project):

```bash
uv init my-shop          # ≈ dotnet new console -o my-shop
cd my-shop
uv add httpx             # ≈ dotnet add package ...   (updates pyproject.toml + uv.lock + .venv)
uv add --dev pytest      # dev-only dependency
uv run main.py           # ≈ dotnet run
uv run pytest            # ≈ dotnet test (pytest is the de-facto test runner)
uv tool install ruff     # ≈ dotnet tool install -g ...
uvx ruff check .         # one-off run of a tool
```

After cloning a repository, a single `uv sync` (≈ `dotnet restore`) recreates `.venv` exactly as described by `uv.lock`.

## Dependency groups

Dependency groups keep tooling out of your runtime dependencies — this project's `dev` group holds `nbconvert` and `nbformat`, which are only needed to build and test the notebooks.

| Command | Effect |
|---|---|
| `uv add --dev pytest` | add to the `dev` group |
| `uv add --group docs mkdocs` | add to a custom `docs` group |
| `uv sync` | installs project deps **+ the `dev` group** (by default) |
| `uv sync --no-dev` | production-style install without dev tools |
| `uv sync --group docs` | also include the `docs` group |

This is similar to a C# solution where test-only and analyzer packages live in a test project or carry `PrivateAssets="all"`, so they never flow to consumers.

## Summary

| Concept | .NET | Python + uv |
|---|---|---|
| Project file | `.csproj` | `pyproject.toml` |
| Lock file | `packages.lock.json` | `uv.lock` |
| Runtime pin | `global.json` | `.python-version` |
| Package registry | nuget.org | PyPI |
| Per-project install location | `bin/`, `obj/` + global cache | `.venv/` + uv cache |
| New project | `dotnet new` | `uv init` |
| Add package | `dotnet add package` | `uv add` |
| Restore | `dotnet restore` | `uv sync` |
| Run | `dotnet run` | `uv run` |
| Global tools | `dotnet tool` | `uv tool` / `uvx` |
| Dev-only deps | `PrivateAssets` / test projects | dependency groups (`--dev`, `--group`) |
| Inspect at runtime | `Environment.Version`, assembly metadata | `sys.version_info`, `sys.prefix`, `importlib.metadata` |

## Exercises

1. Write a function `env_report()` that returns a dict with the Python version as a `"3.14.x"` string, whether a virtual environment is active, and the number of installed distributions.
2. Write `installed_version(name: str) -> str | None` that returns the installed version of a package, or `None` if it isn't installed. Test it with `"nbformat"` and `"not-a-real-package"`.
3. Which uv command would you use for each: (a) a teammate cloned the repo and wants the environment; (b) you need `pytest` only for development; (c) you want to try the `ruff` linter once without installing it?

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import sys
from importlib import metadata

# 1
def env_report() -> dict:
    v = sys.version_info
    return {
        "python": f"{v.major}.{v.minor}.{v.micro}",
        "in_venv": sys.prefix != sys.base_prefix,
        "distributions": len(list(metadata.distributions())),
    }

print(env_report())

# 2
def installed_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None

print(installed_version("nbformat"), installed_version("not-a-real-package"))

# 3
# (a) uv sync               ≈ dotnet restore
# (b) uv add --dev pytest   ≈ a dev-only / PrivateAssets reference
# (c) uvx ruff check .      ≈ dotnet tool exec / dnx
```

</details>